# GraDeT-HTR: Bengali Handwritten Text Recognition - Training Notebook

This notebook trains the GraDeT-HTR model (Decoder-only Transformer with ViT patch embeddings + GPT-2 style blocks) for Bengali handwritten text recognition.

**Architecture**: ViT Patch Embeddings (image encoder) + Custom Bengali Grapheme Tokenizer + 12-layer GPT-2 Transformer Decoder

**Two training modes**:
1. **Standard mode** - Train on image-text pairs
2. **Context-aware mode** - Use previous word context to improve recognition

## 1. Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone Repository and Install Dependencies

In [ ]:
import os

# Clone the repository (skip if already cloned)
if not os.path.isdir("/content/GraDeT-HTR"):
    !git clone https://github.com/shyan23/GraDeT-HTR_context.git /content/GraDeT-HTR
    # If the repo is private, use a personal access token:
    # !git clone https://<YOUR_TOKEN>@github.com/shyan23/GraDeT-HTR_context.git /content/GraDeT-HTR
else:
    print("Repository already exists at /content/GraDeT-HTR, skipping clone.")
    # Pull latest changes
    !cd /content/GraDeT-HTR && git pull

In [ ]:
# Clear cached/stale bytecode and Colab notebook cache
# Preserves: model weights (.pth), training data, and the repo itself
import os, glob, shutil

# 1. Remove all __pycache__ directories in the repo
for root, dirs, files in os.walk("/content/GraDeT-HTR"):
    for d in dirs:
        if d == "__pycache__":
            path = os.path.join(root, d)
            shutil.rmtree(path)
            print(f"Removed: {path}")

# 2. Remove .pyc files from installed packages (transformers etc.)
!find /usr/local/lib/python3.*/dist-packages -name "*.pyc" -delete 2>/dev/null; true

# 3. Clear Colab's IPython input cache (prevents stale cell reruns)
!rm -rf /root/.cache/ipython 2>/dev/null; true
!rm -rf /tmp/ipython-input-* 2>/dev/null; true

# 4. Clear Python's import cache so modules reload fresh
import importlib
importlib.invalidate_caches()

print("\nCache cleared. Model weights and training data are preserved.")

In [ ]:
# Install core dependencies
# Only install and restart if transformers is not the correct version
import importlib, subprocess, sys

_need_restart = False
try:
    import transformers
    if transformers.__version__ != "4.48.0":
        raise ImportError("wrong version")
    print(f"transformers {transformers.__version__} already installed, skipping.")
except (ImportError, AttributeError):
    _need_restart = True
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.0"])

# Install other deps (idempotent, won't trigger restart)
!pip install -q marisa-trie==1.2.1
!pip install -q git+https://github.com/csebuetnlp/normalizer@d80c3c4#egg=normalizer
!pip install -q albumentations==2.0.5

if _need_restart:
    print("\n--- Restarting runtime to load new transformers version ---")
    print("After restart, re-run this cell (it will skip the install).")
    import os
    os.kill(os.getpid(), 9)
else:
    print("All dependencies ready, no restart needed.")

## 3. Upload Your Training Data

You have **three options** to get your data into Colab:

### Option A: Google Drive (Recommended for large datasets)
Upload your training data using `GlyphScribe/upload_to_gdrive.py`, which places `training_data.zip` in the `GraDeT-HTR-Training` folder on Google Drive.

In [ ]:
# Download training data from Google Drive
!pip install -q gdown
!gdown 1txYG1JoP62RWWrn_0FxCQ92g9pvYNFWl -O /content/grade-htr-training.zip

# Unzip training data
!unzip -q /content/grade-htr-training.zip -d /content/

# Set paths (zip extracts to training/ directory)
IMAGES_DIR = "/content/training/images"
LABELS_FILE = "/content/training/labels/label.csv"

# For context-aware mode (Stage 3):
JSON_DIR = "/content/training/json"

# --- Verify training data setup ---
import os
assert os.path.isdir(IMAGES_DIR), f"Images directory not found: {IMAGES_DIR}"
assert os.path.isfile(LABELS_FILE), f"Labels file not found: {LABELS_FILE}"

num_images = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
import csv
with open(LABELS_FILE, encoding='utf-8') as f:
    num_labels = sum(1 for _ in csv.reader(f)) - 1  # subtract header

print(f"Images found: {num_images}")
print(f"Labels found: {num_labels}")
if num_images != num_labels:
    print(f"WARNING: mismatch — {num_images} images vs {num_labels} labels")
else:
    print("Data setup verified OK")

In [ ]:
# Option B: Use the sample data included in the repo (for testing/demo)
# DISABLED — Using Google Drive data (Option A) instead
# IMAGES_DIR = "/content/GraDeT-HTR/sample_train/images"
# LABELS_FILE = "/content/GraDeT-HTR/sample_train/labels/label.csv"
# JSON_DIR = None

In [ ]:
# Option C: Upload a zip file directly
# from google.colab import files
# uploaded = files.upload()  # Upload your data.zip
# !unzip -q data.zip -d /content/training_data/
# IMAGES_DIR = "/content/training_data/images"
# LABELS_FILE = "/content/training_data/labels/label.csv"

## 4. Verify Data

In [ ]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Verify data exists
print(f"Images directory exists: {os.path.exists(IMAGES_DIR)}")
print(f"Labels file exists: {os.path.exists(LABELS_FILE)}")

if os.path.exists(IMAGES_DIR):
    num_images = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"Number of images: {num_images}")

if os.path.exists(LABELS_FILE):
    df = pd.read_csv(LABELS_FILE)
    print(f"Number of labels: {len(df)}")
    print(f"\nSample labels:")
    print(df.head(10))

In [ ]:
# Visualize a few sample images with their labels
if os.path.exists(IMAGES_DIR) and os.path.exists(LABELS_FILE):
    n_show = min(5, len(df))
    fig, axes = plt.subplots(1, n_show, figsize=(15, 3))
    if n_show == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        if i >= len(df):
            break
        img_path = os.path.join(IMAGES_DIR, str(df.iloc[i]['image_id']))
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(str(df.iloc[i]['text']), fontsize=10)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Setup Paths and Imports

In [ ]:
import sys
import os

# Add project directories to Python path
REPO_ROOT = "/content/GraDeT-HTR"
sys.path.insert(0, os.path.join(REPO_ROOT, "GraDeT_HTR"))
sys.path.insert(0, REPO_ROOT)

# Change to GraDeT_HTR directory (needed for relative path resolution)
os.chdir(os.path.join(REPO_ROOT, "GraDeT_HTR"))
print(f"Working directory: {os.getcwd()}")

In [ ]:
import torch
import tqdm
from torch.utils.data import DataLoader

from config import DTrOCRConfig
from model import DTrOCRLMHeadModel
from dataset import split_data, split_context_data, HandwrittenDataset
from utils import (
    send_inputs_to_device, evaluate_model,
    save_checkpoint, save_final_model, load_checkpoint
)

print("All imports successful!")

## 6. Configuration

In [ ]:
#============================================================
# TRAINING CONFIGURATION - Modify these as needed
#============================================================

# Training mode
CONTEXT_AWARE = False          # Set True for context-aware training

# Multi-stage training (per EMNLP paper):
#   Stage 1: Line-level pretraining (LR=1e-4)
#   Stage 2: Word-level pretraining (LR=1e-4)
#   Stage 3: Context-aware fine-tuning (LR=5e-6, CONTEXT_AWARE=True)
TRAINING_STAGE = 2             # 1=line pretrain, 2=word pretrain, 3=finetune

# Stage-aware hyperparameters
BATCH_SIZE = 96                # L4 24GB can handle 96 (was 32); reduce if OOM
EPOCHS = 10
if TRAINING_STAGE <= 2:
    LEARNING_RATE = 1e-4       # Pretraining LR
else:
    LEARNING_RATE = 5e-6       # Fine-tuning LR (per paper)
    CONTEXT_AWARE = True       # Stage 3 requires context-aware mode
MARGIN_LAMBDA = 0.5            # Only used in context-aware mode

# Previous stage checkpoint (for stage 2 or 3, load from previous stage's final model)
# Set to None for stage 1 or training from scratch
PREV_STAGE_MODEL = None
# PREV_STAGE_MODEL = "/content/GraDeT-HTR/output/final_model.pth"  # Uncomment for stage 2/3

# Output directory for checkpoints and final model
OUTPUT_DIR = "/content/GraDeT-HTR/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mount Google Drive for backup saves
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/GraDeT-HTR-Training/output"
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Training Stage: {TRAINING_STAGE}")
print(f"  Mode: {'Context-Aware' if CONTEXT_AWARE else 'Standard'}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Images: {IMAGES_DIR}")
print(f"  Labels: {LABELS_FILE}")
print(f"  Output (local): {OUTPUT_DIR}")
print(f"  Output (GDrive): {GDRIVE_OUTPUT_DIR}")
if PREV_STAGE_MODEL:
    print(f"  Loading from: {PREV_STAGE_MODEL}")

## 7. Initialize Model and Data

In [ ]:
  # Clear all Colab/IPython cached cell inputs and bytecode                                                        
  !rm -rf /tmp/ipython-input-* 2>/dev/null
  !rm -rf /root/.cache/ipython 2>/dev/null                                                                         
  !rm -rf /root/.local/share/jupyter 2>/dev/null                                                                   
  !find /content/GraDeT-HTR -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null
  !find /content/GraDeT-HTR -name "*.pyc" -delete 2>/dev/null

  # Force Python to forget all cached imports
  import importlib, sys
  importlib.invalidate_caches()
  # Remove any previously loaded project modules so they reload fresh
  for mod_name in list(sys.modules.keys()):
      if any(x in mod_name for x in ['config', 'model', 'dataset', 'utils', 'bntokenizer', 'processor',
  'BnGraphemizer']):
          del sys.modules[mod_name]

  print("All caches cleared.")


In [ ]:
# Vocabulary file path
import os
REPO_ROOT = "/content/GraDeT-HTR"
VOCAB_FILE = os.path.join(
    REPO_ROOT, "tokenization",
    "bn_grapheme_1296_from_bengali.ai.buet.txt"
)
print(f"Vocab file exists: {os.path.exists(VOCAB_FILE)}")

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
torch.set_float32_matmul_precision('high')

# Model config
config = DTrOCRConfig(bn_vocab_file=VOCAB_FILE)

In [ ]:
# Build datasets
if CONTEXT_AWARE:
    print("Building context-aware datasets...")
    train_dataset, val_dataset = split_context_data(
        IMAGES_DIR, JSON_DIR, config, test_size=0.1
    )
else:
    print("Building standard datasets...")
    train_dataset, val_dataset = split_data(
        IMAGES_DIR, LABELS_FILE, config
    )

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Create dataloaders (pin_memory + workers for faster GPU transfer)
train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

In [ ]:
# Initialize model
model = DTrOCRLMHeadModel(config)

# Load previous stage weights if specified
if PREV_STAGE_MODEL and os.path.exists(PREV_STAGE_MODEL):
    from utils import load_final_model
    model = load_final_model(model, PREV_STAGE_MODEL)
    print(f"Loaded weights from previous stage: {PREV_STAGE_MODEL}")

model.to(device)

# Compile model for faster training (L4 supports this well)
model = torch.compile(model)
print("torch.compile() enabled")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Optimizer and AMP setup
use_amp = True
# L4 GPU supports bf16 natively — more stable and faster than fp16
amp_dtype = torch.bfloat16
scaler = torch.amp.GradScaler(device=device, enabled=False)  # bf16 doesn't need scaler
optimizer = torch.optim.AdamW(
    params=model.parameters(), lr=LEARNING_RATE, weight_decay=0.01
)

print("Optimizer: AdamW (weight_decay=0.01)")
print(f"Mixed precision: bf16 (native L4 support)")

## 8. (Optional) Resume from Checkpoint

In [ ]:
#Uncomment to resume from a checkpoint
CHECKPOINT_PATH = "/content/GraDeT-HTR/output/checkpoint_epoch_5.pt"
model, optimizer, start_epoch = load_checkpoint(
    CHECKPOINT_PATH, model, optimizer
)
model.to(device)
print(f"Resumed from epoch {start_epoch}")

start_epoch = 0  # Set this to the checkpoint epoch if resuming

## 9. Training Loop

In [ ]:
def evaluate_context_model(model, dataloader, device, use_amp=True):
    """Evaluate model with context-aware dual-path loss."""
    model.eval()
    val_losses, val_ctx_losses, val_iso_losses = [], [], []
    val_accs = []

    with torch.no_grad():
        for inputs in dataloader:
            pixel_values = inputs['pixel_values'].to(device)
            labels = inputs['labels'].to(device)

            input_ids_ctx = inputs['input_ids'].to(device)
            attention_mask_ctx = inputs['attention_mask'].to(device)
            ctx_len = inputs['context_length']
            if isinstance(ctx_len, torch.Tensor):
                ctx_len = ctx_len[0].item()

            input_ids_iso = inputs['input_ids_isolated'].to(device)
            attention_mask_iso = inputs['attention_mask_isolated'].to(device)
            labels_iso = inputs['labels_isolated'].to(device)
            ctx_len_iso = inputs['context_length_isolated']
            if isinstance(ctx_len_iso, torch.Tensor):
                ctx_len_iso = ctx_len_iso[0].item()

            with torch.autocast(device_type=device, dtype=amp_dtype):
                outputs_ctx = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_ctx,
                    attention_mask=attention_mask_ctx,
                    labels=labels,
                    context_length=ctx_len
                )
                outputs_iso = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_iso,
                    attention_mask=attention_mask_iso,
                    labels=labels_iso,
                    context_length=ctx_len_iso
                )

            loss_net = outputs_ctx.loss + outputs_iso.loss
            val_losses.append(loss_net.item())
            val_ctx_losses.append(outputs_ctx.loss.item())
            val_iso_losses.append(outputs_iso.loss.item())
            val_accs.append(
                (outputs_ctx.accuracy.item() + outputs_iso.accuracy.item()) / 2
            )

    avg = lambda lst: sum(lst) / len(lst)
    return avg(val_losses), avg(val_accs), avg(val_ctx_losses), avg(val_iso_losses)

In [ ]:
# Training loop
train_losses, train_accuracies = [], []
validation_losses, validation_accuracies = [], []

if CONTEXT_AWARE:
    print("\n=== Training with Context-Aware Dual-Path Loss ===")
    print(f"L = -log(Pc) - log(Pi) + {MARGIN_LAMBDA}*max(0, loss_ctx - loss_iso)\n")

for epoch in range(start_epoch, EPOCHS):
    epoch_losses, epoch_accuracies = [], []
    epoch_ctx_losses, epoch_iso_losses = [], []
    epoch_ctx_accs, epoch_iso_accs = [], []
    epoch_margin_terms = []

    model.train()
    progress_bar = tqdm.tqdm(
        train_dataloader,
        total=len(train_dataloader),
        desc=f'Epoch {epoch + 1}/{EPOCHS}'
    )

    for inputs in progress_bar:
        optimizer.zero_grad()

        pixel_values = inputs['pixel_values'].to(device)
        labels = inputs['labels'].to(device)

        if CONTEXT_AWARE:
            # === DUAL-PATH LOSS WITH MARGIN TERM ===
            input_ids_ctx = inputs['input_ids'].to(device)
            attention_mask_ctx = inputs['attention_mask'].to(device)
            ctx_len = inputs['context_length']
            if isinstance(ctx_len, torch.Tensor):
                ctx_len = ctx_len[0].item()

            input_ids_iso = inputs['input_ids_isolated'].to(device)
            attention_mask_iso = inputs['attention_mask_isolated'].to(device)
            ctx_len_iso = inputs['context_length_isolated']
            if isinstance(ctx_len_iso, torch.Tensor):
                ctx_len_iso = ctx_len_iso[0].item()
            labels_iso = inputs['labels_isolated'].to(device)

            with torch.autocast(device_type=device, dtype=amp_dtype):
                outputs_ctx = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_ctx,
                    attention_mask=attention_mask_ctx,
                    labels=labels,
                    context_length=ctx_len,
                    return_per_sample_loss=True
                )
                outputs_iso = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_iso,
                    attention_mask=attention_mask_iso,
                    labels=labels_iso,
                    context_length=ctx_len_iso,
                    return_per_sample_loss=True
                )

            combined_base_loss = (
                outputs_ctx.per_sample_loss + outputs_iso.per_sample_loss
            )
            margin_term = MARGIN_LAMBDA * torch.clamp(
                outputs_ctx.per_sample_loss - outputs_iso.per_sample_loss,
                min=0.0
            )
            weighted_loss = (combined_base_loss + margin_term).mean()

            weighted_loss.backward()
            optimizer.step()

            epoch_losses.append(weighted_loss.item())
            epoch_ctx_losses.append(outputs_ctx.loss.item())
            epoch_iso_losses.append(outputs_iso.loss.item())
            epoch_ctx_accs.append(outputs_ctx.accuracy.item())
            epoch_iso_accs.append(outputs_iso.accuracy.item())
            epoch_accuracies.append(
                (outputs_ctx.accuracy.item() + outputs_iso.accuracy.item()) / 2
            )
            epoch_margin_terms.append(margin_term.mean().item())

            progress_bar.set_postfix({
                'loss': f'{weighted_loss.item():.4f}',
                'ctx_acc': f'{outputs_ctx.accuracy.item():.4f}',
                'iso_acc': f'{outputs_iso.accuracy.item():.4f}'
            })
        else:
            # === STANDARD TRAINING ===
            input_ids = inputs['input_ids'].to(device)
            attention_mask = inputs['attention_mask'].to(device)

            with torch.autocast(device_type=device, dtype=amp_dtype):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            outputs.loss.backward()
            optimizer.step()

            epoch_losses.append(outputs.loss.item())
            epoch_accuracies.append(outputs.accuracy.item())

            progress_bar.set_postfix({
                'loss': f'{outputs.loss.item():.4f}',
                'acc': f'{outputs.accuracy.item():.4f}'
            })

    # Epoch metrics
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    avg_acc = sum(epoch_accuracies) / len(epoch_accuracies)
    train_losses.append(avg_loss)
    train_accuracies.append(avg_acc)

    # Validation
    if CONTEXT_AWARE:
        val_loss, val_acc, val_ctx_loss, val_iso_loss = (
            evaluate_context_model(model, val_dataloader, device, use_amp)
        )
        validation_losses.append(val_loss)
        validation_accuracies.append(val_acc)

        avg_margin = (
            sum(epoch_margin_terms) / len(epoch_margin_terms)
            if epoch_margin_terms else 0.0
        )
        print(f"\nEpoch {epoch + 1}/{EPOCHS}:")
        print(f"  Train - Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")
        ctx_l = sum(epoch_ctx_losses) / len(epoch_ctx_losses)
        iso_l = sum(epoch_iso_losses) / len(epoch_iso_losses)
        ctx_a = sum(epoch_ctx_accs) / len(epoch_ctx_accs)
        iso_a = sum(epoch_iso_accs) / len(epoch_iso_accs)
        print(f"    Ctx Loss: {ctx_l:.4f} | Iso Loss: {iso_l:.4f}")
        print(f"    Ctx Acc: {ctx_a:.4f} | Iso Acc: {iso_a:.4f}")
        print(f"    Margin Mean: {avg_margin:.4f}")
        print(f"  Val - Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
        print(f"    Ctx Loss: {val_ctx_loss:.4f} | Iso Loss: {val_iso_loss:.4f}")
    else:
        val_loss, val_acc = evaluate_model(
            model, val_dataloader, device=device
        )
        validation_losses.append(val_loss)
        validation_accuracies.append(val_acc)

        print(
            f"\nEpoch {epoch + 1}/{EPOCHS}: "
            f"Train Loss: {avg_loss:.4f}, "
            f"Train Acc: {avg_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_acc:.4f}"
        )

    # Save checkpoint to both local and Google Drive
    import shutil
    mode_prefix = "context_" if CONTEXT_AWARE else ""
    ckpt_name = f'checkpoint_{mode_prefix}epoch_{epoch+1}.pt'
    save_checkpoint(
        model, optimizer, epoch + 1,
        train_losses[-1], validation_losses[-1],
        train_accuracies[-1], validation_accuracies[-1],
        OUTPUT_DIR, ckpt_name
    )
    # Copy checkpoint to Google Drive
    local_ckpt = os.path.join(OUTPUT_DIR, ckpt_name)
    gdrive_ckpt = os.path.join(GDRIVE_OUTPUT_DIR, ckpt_name)
    shutil.copy2(local_ckpt, gdrive_ckpt)
    print(f"  Checkpoint saved: {local_ckpt} + {gdrive_ckpt}")

print("\nTraining complete!")

## 10. Save Final Model

In [ ]:
import shutil

# Save the final model
mode_prefix = "context_" if CONTEXT_AWARE else ""
final_model_name = f'final_{mode_prefix}stage{TRAINING_STAGE}_model.pth'
final_model_path = os.path.join(OUTPUT_DIR, final_model_name)
save_final_model(model, final_model_path)
print(f"Final model saved (local): {final_model_path}")

# Copy to Google Drive
gdrive_final_path = os.path.join(GDRIVE_OUTPUT_DIR, final_model_name)
shutil.copy2(final_model_path, gdrive_final_path)
print(f"Final model saved (GDrive): {gdrive_final_path}")

print(f"\nFor next stage, set PREV_STAGE_MODEL = \"{final_model_path}\"")

## 11. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(train_losses) + 1)

# Loss plot
ax1.plot(epochs_range, train_losses, 'b-o', label='Train Loss')
ax1.plot(epochs_range, validation_losses, 'r-o', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(epochs_range, train_accuracies, 'b-o', label='Train Accuracy')
ax2.plot(epochs_range, validation_accuracies, 'r-o', label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {plot_path}")

# Also save plot to Google Drive
import shutil
shutil.copy2(plot_path, os.path.join(GDRIVE_OUTPUT_DIR, 'training_curves.png'))
print(f"Plot also saved to GDrive")

## 12. Download Trained Model

In [ ]:
import shutil
from google.colab import files

# Copy all outputs to Google Drive (already done per-epoch, but ensure final state)
print("Syncing all outputs to Google Drive...")
for f_name in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, f_name)
    dst = os.path.join(GDRIVE_OUTPUT_DIR, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"All files synced to: {GDRIVE_OUTPUT_DIR}")

# Download final model locally to your machine
print(f"\nDownloading {final_model_name} to your local machine...")
files.download(final_model_path)

## 13. (Optional) Quick Inference Test

In [ ]:
from processor import DTrOCRProcessor
from utils import load_final_model
from PIL import Image
import matplotlib.pyplot as plt

# Load the trained model for inference
inference_model = DTrOCRLMHeadModel(config)
inference_model = load_final_model(inference_model, final_model_path)
inference_model.to(device)
inference_model.eval()

# Create processor (BOS only, no EOS for generation)
processor = DTrOCRProcessor(
    config, add_bos_token=True, add_eos_token=False
)

# Test on a sample image
test_images = [
    f for f in os.listdir(IMAGES_DIR)
    if f.endswith(('.jpg', '.png', '.jpeg'))
]

if test_images:
    test_img_path = os.path.join(IMAGES_DIR, test_images[0])
    test_img = Image.open(test_img_path).convert('RGB')

    # Process image
    inputs = processor(
        images=test_img, texts="",
        padding=True, return_tensors="pt"
    )

    # Move to device
    inputs.pixel_values = inputs.pixel_values.to(device)
    inputs.input_ids = inputs.input_ids.to(device)
    inputs.attention_mask = inputs.attention_mask.to(device)

    # Generate
    output_ids = inference_model.generate(
        inputs, processor, num_beams=1
    )

    # Decode
    predicted_text = processor.tokeniser.bn_graphmemizer.ids_to_text(
        output_ids[0].cpu().tolist()
    )

    plt.figure(figsize=(6, 2))
    plt.imshow(test_img)
    plt.title(f"Predicted: {predicted_text}")
    plt.axis('off')
    plt.show()
else:
    print("No test images found.")

---

## Stage 3: Context-Aware Fine-Tuning with RoPE

Stage 2 used **absolute positional embeddings** (`nn.Embedding(256)`). In context mode, the sequence becomes `[context(20) + patches(128) + target(128)] = 276 > 256`, causing a CUDA overflow.

**Solution**: Switch to **Rotary Position Embeddings** (RoPE, Su et al. 2023). Instead of additive absolute embeddings, RoPE rotates Q and K vectors inside attention:

$$f_{\{q,k\}}(x_m, m) = R^d_{\Theta,m} W_{\{q,k\}} x_m$$

This encodes **relative position** via rotation, with no fixed sequence length limit. The efficient computation (Eq. 34 from the paper):

$$R^d_{\Theta,m} x = x \otimes \cos(m\Theta) + \text{rotate\_half}(x) \otimes \sin(m\Theta)$$

**Dual-path context loss** (same as before):

$$L = -\log P_c - \log P_i + \lambda \cdot \max(0, \ell_c - \ell_i)$$

### 14. Apply RoPE Architecture from Branch

In [ ]:
# Fetch the RoPE branch and checkout its architecture files
# Adds: rope.py, rope_attention.py
# Patches: model.py (RoPE support), config.py (use_rope flag),
#          dataset.py (process image once), utils.py (strict param)
!cd /content/GraDeT-HTR && git fetch origin RoPE
!cd /content/GraDeT-HTR && git checkout origin/RoPE -- \
    GraDeT_HTR/rope.py \
    GraDeT_HTR/rope_attention.py \
    GraDeT_HTR/model.py \
    GraDeT_HTR/config.py \
    GraDeT_HTR/dataset.py \
    GraDeT_HTR/utils.py

print("RoPE files applied. Verifying...")
import os
for f in ['rope.py', 'rope_attention.py']:
    path = os.path.join(REPO_ROOT, 'GraDeT_HTR', f)
    print(f"  {f}: {'OK' if os.path.exists(path) else 'MISSING'}")

# --- Fix processor.py: empty context must still prepend padding block ---
# Without this fix, samples with empty prev_text get input_ids of length 128,
# while samples with context get length 148, causing collation crash.
proc_path = os.path.join(REPO_ROOT, 'GraDeT_HTR', 'processor.py')
with open(proc_path, 'r') as f:
    proc_code = f.read()

old = '        # If context is provided, prepend it with separator and pad to fixed length\n        if context_text is not None and context_text != "" and text_inputs is not None:\n            # Tokenize context (no padding — we pad to fixed length ourselves)\n            context_inputs = self.tokeniser(\n                context_text, padding=False\n            )\n\n            sep_token_id = self.tokeniser.eos_token_id  # Reuse EOS as separator\n            pad_token_id = self.tokeniser.pad_token_id\n            context_ids = context_inputs[\'input_ids\']'

new = '      
# Find and add the else branch for empty context
old2 = '            context_attn_mask = [1] * actual_len + [0] * pad_needed\n\n            # Fixed context_length for ALL samples (enables uniform batching)\n            context_length = self.max_context_length'

new2 = '                context_attn_mask = [1] * actual_len + [0] * pad_needed\n            else:\n                # Empty context: all padding, fully masked out\n                context_padded = [pad_token_id] * self.max_context_length\n                context_attn_mask = [0] * self.max_context_length\n\n            # Fixed context_length for ALL samples (enables uniform batching)\n            context_length = self.max_context_length'

if old in proc_code:
    proc_code = proc_code.replace(old, new)
    # Also need to indent the block between the new inner if and the else
    # The lines between context_ids and context_attn_mask need extra indent
    proc_code = proc_code.replace(old2, new2)
    with open(proc_path, 'w') as f:
        f.write(proc_code)
    print("\nprocessor.py patched: empty context now gets padding block")
else:
    print("\nprocessor.py already patched or has different content, skipping")  # If context_text is provided (even empty), prepend a fixed-length context block\n        # so all samples have identical sequence length for batching\n        if context_text is not None and text_inputs is not None:\n            pad_token_id = self.tokeniser.pad_token_id\n\n            if context_text != "":\n                # Tokenize context (no padding — we pad to fixed length ourselves)\n                context_inputs = self.tokeniser(\n                    context_text, padding=False\n                )\n\n                sep_token_id = self.tokeniser.eos_token_id  # Reuse EOS as separator\n                context_ids = context_inputs[\'input_ids\']'


In [ ]:
# Reload all changed modules so Python picks up the RoPE code + processor fix
import importlib
import config, model, dataset, utils, processor
importlib.reload(config)
importlib.reload(model)
importlib.reload(dataset)
importlib.reload(utils)
importlib.reload(processor)

from config import DTrOCRConfig
from model import DTrOCRLMHeadModel
from dataset import split_context_data, ContextAwareDataset
from utils import save_checkpoint, save_final_model, load_final_model, load_checkpoint
from processor import DTrOCRProcessor

print("Modules reloaded with RoPE support + processor fix.")

### 15. Stage 3 Configuration

In [ ]:
#============================================================
# STAGE 3 CONFIGURATION — RoPE + Context-Aware
#============================================================

TRAINING_STAGE = 3
CONTEXT_AWARE = True

# Fine-tuning hyperparameters
LEARNING_RATE_S3 = 5e-6        # Much lower LR for fine-tuning
BATCH_SIZE_S3 = 32             # Reduce if OOM (each step does 2 forward passes)
EPOCHS_S3 = 10
MARGIN_LAMBDA = 0.5            # Weight for max(0, loss_ctx - loss_iso) penalty

# Data paths
JSON_DIR = "/content/training/json"

# Stage 2 checkpoint (absolute PE model)
STAGE2_MODEL = "/content/GraDeT-HTR/output/final_stage2_model.pth"

# Output directory
OUTPUT_DIR_S3 = "/content/GraDeT-HTR/output"
os.makedirs(OUTPUT_DIR_S3, exist_ok=True)

print("Stage 3 Configuration:")
print(f"  Mode: RoPE + Context-Aware Dual-Path")
print(f"  Batch size: {BATCH_SIZE_S3}")
print(f"  Epochs: {EPOCHS_S3}")
print(f"  Learning rate: {LEARNING_RATE_S3}")
print(f"  Margin lambda: {MARGIN_LAMBDA}")
print(f"  JSON dir: {JSON_DIR}")
print(f"  Loading Stage 2 from: {STAGE2_MODEL}")

### 16. Verify Context Data (JSON files)

In [ ]:
import json, glob

assert os.path.isdir(JSON_DIR), f"JSON directory not found: {JSON_DIR}"

json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
print(f"JSON files found: {len(json_files)}")

print("\nSample JSON entries:")
for jf in json_files[:3]:
    with open(jf, encoding='utf-8') as f:
        data = json.load(f)
    print(f"  {os.path.basename(jf)}: text='{data.get('text', 'MISSING')}', "
          f"line_id={data.get('line_id', 'MISSING')}, "
          f"word_index={data.get('word_index', 'MISSING')}")

with open(json_files[0], encoding='utf-8') as f:
    sample = json.load(f)
required_fields = ['text', 'output_path', 'line_id', 'word_index']
missing = [k for k in required_fields if k not in sample]
if missing:
    print(f"\nWARNING: Missing required fields: {missing}")
else:
    print("\nAll required fields present.")

### 17. Build Context-Aware Datasets

In [ ]:
# Build context-aware datasets (splits by line_id to keep sentences intact)
# Note: uses the patched dataset.py that processes images once per sample
print("Building context-aware datasets...")

# Config for dataset — use_rope doesn't affect data loading
config_s3 = DTrOCRConfig(bn_vocab_file=VOCAB_FILE, use_rope=True)

train_dataset_ctx, val_dataset_ctx = split_context_data(
    IMAGES_DIR, JSON_DIR, config_s3, test_size=0.1
)

print(f"Train samples: {len(train_dataset_ctx)}")
print(f"Validation samples: {len(val_dataset_ctx)}")

train_dataloader_ctx = DataLoader(
    train_dataset_ctx, batch_size=BATCH_SIZE_S3,
    shuffle=True, num_workers=0
)
val_dataloader_ctx = DataLoader(
    val_dataset_ctx, batch_size=BATCH_SIZE_S3,
    shuffle=False, num_workers=0
)

print(f"Train batches: {len(train_dataloader_ctx)}")
print(f"Validation batches: {len(val_dataloader_ctx)}")

In [ ]:
# Inspect a batch to understand the dual-path structure
sample_batch = next(iter(train_dataloader_ctx))

print("Batch keys:", list(sample_batch.keys()))
print(f"\nShared:")
print(f"  pixel_values:            {sample_batch['pixel_values'].shape}")

print(f"\nContext path (prev word prepended):")
print(f"  input_ids:               {sample_batch['input_ids'].shape}")
print(f"  attention_mask:          {sample_batch['attention_mask'].shape}")
print(f"  labels:                  {sample_batch['labels'].shape}")
print(f"  context_length:          {sample_batch['context_length']}")

print(f"\nIsolation path (no context):")
print(f"  input_ids_isolated:      {sample_batch['input_ids_isolated'].shape}")
print(f"  attention_mask_isolated: {sample_batch['attention_mask_isolated'].shape}")
print(f"  labels_isolated:         {sample_batch['labels_isolated'].shape}")
print(f"  context_length_isolated: {sample_batch['context_length_isolated']}")

print(f"\nMetadata:")
print(f"  prev_text[0]: '{sample_batch['prev_text'][0]}'")
print(f"  text[0]:      '{sample_batch['text'][0]}'")

### 18. Convert Stage 2 Weights (GPT2 → RoPE) and Initialize Model

Stage 2 used `GPT2Block` with `Conv1D` layers. RoPE uses `RoPEBlock` with `nn.Linear`.
We need to:
1. Skip `positional_embedding` (RoPE computes positions on the fly)
2. Split combined `c_attn` (QKV) → separate `q_proj`, `k_proj`, `v_proj`
3. Transpose Conv1D weights to Linear format (`Conv1D.weight.T`)

In [ ]:
import torch

def convert_gpt2_to_rope(state_dict):
    """
    Convert Stage 2 state_dict (GPT2Block + absolute PE) to RoPE format.

    Key changes:
    - Drop positional_embedding (RoPE has no additive position embedding)
    - Split attn.c_attn (combined QKV Conv1D) -> q_proj, k_proj, v_proj (Linear)
    - Transpose Conv1D weights -> Linear weights (Conv1D stores [in, out], Linear stores [out, in])
    - Skip causal mask buffers (attn.bias, attn.masked_bias)
    """
    new_sd = {}
    skipped, converted = [], []

    for key, value in state_dict.items():
        # Strip torch.compile prefix
        key = key.replace('_orig_mod.', '') if key.startswith('_orig_mod.') else key

        # Skip absolute positional embedding
        if 'positional_embedding' in key:
            skipped.append(key)
            continue

        # Skip causal mask buffers
        if key.endswith('.attn.bias') or key.endswith('.attn.masked_bias'):
            skipped.append(key)
            continue

        # --- Attention: split combined c_attn into separate Q, K, V ---
        if '.attn.c_attn.weight' in key:
            prefix = key.replace('.attn.c_attn.weight', '')
            # Conv1D shape: [in_features, 3*out_features]
            q, k, v = value.split(value.shape[1] // 3, dim=1)
            new_sd[f'{prefix}.attn.q_proj.weight'] = q.T
            new_sd[f'{prefix}.attn.k_proj.weight'] = k.T
            new_sd[f'{prefix}.attn.v_proj.weight'] = v.T
            converted.append(key + ' -> q/k/v_proj.weight')
            continue

        if '.attn.c_attn.bias' in key:
            prefix = key.replace('.attn.c_attn.bias', '')
            q, k, v = value.split(value.shape[0] // 3, dim=0)
            new_sd[f'{prefix}.attn.q_proj.bias'] = q
            new_sd[f'{prefix}.attn.k_proj.bias'] = k
            new_sd[f'{prefix}.attn.v_proj.bias'] = v
            converted.append(key + ' -> q/k/v_proj.bias')
            continue

        # --- Attention output projection: Conv1D -> Linear ---
        if '.attn.c_proj.weight' in key:
            new_sd[key] = value.T
            converted.append(key + ' (transposed)')
            continue

        # --- MLP: Conv1D -> Linear ---
        if '.mlp.c_fc.weight' in key or '.mlp.c_proj.weight' in key:
            new_sd[key] = value.T
            converted.append(key + ' (transposed)')
            continue

        # Everything else transfers directly
        new_sd[key] = value

    print(f"Skipped {len(skipped)} keys: {skipped}")
    print(f"Converted {len(converted)} keys:")
    for c in converted:
        print(f"  {c}")
    print(f"Total output keys: {len(new_sd)}")
    return new_sd

print("Weight conversion function defined.")

In [ ]:
# Build RoPE model and load converted Stage 2 weights
model_s3 = DTrOCRLMHeadModel(config_s3)
model_s3.to(device)

# Verify RoPE is active
print(f"use_rope: {config_s3.use_rope}")
print(f"Has rotary_emb: {hasattr(model_s3.transformer, 'rotary_emb')}")
print(f"Has positional_embedding: {hasattr(model_s3.transformer, 'positional_embedding')}")
print(f"Block type: {type(model_s3.transformer.hidden_layers[0]).__name__}")

# Load and convert Stage 2 weights
print(f"\nLoading Stage 2 weights from: {STAGE2_MODEL}")
stage2_sd = torch.load(STAGE2_MODEL, map_location='cpu')
rope_sd = convert_gpt2_to_rope(stage2_sd)

# Load converted weights (strict=False skips rotary_emb.inv_freq buffer)
missing, unexpected = model_s3.load_state_dict(rope_sd, strict=False)
print(f"\nMissing keys (expected — RoPE buffers): {missing}")
print(f"Unexpected keys: {unexpected}")

# Count parameters
total_params = sum(p.numel() for p in model_s3.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Fresh optimizer with low LR for fine-tuning
optimizer_s3 = torch.optim.Adam(model_s3.parameters(), lr=LEARNING_RATE_S3)
scaler_s3 = torch.amp.GradScaler(device=device, enabled=use_amp)
print(f"Optimizer: Adam (lr={LEARNING_RATE_S3})")

### 19. Context-Aware Training Loop

In [ ]:
def evaluate_context_model(model, dataloader, device, use_amp=True):
    """Evaluate with dual-path loss (no margin term during eval)."""
    model.eval()
    val_losses, val_ctx_losses, val_iso_losses = [], [], []
    val_ctx_accs, val_iso_accs = [], []

    with torch.no_grad():
        for inputs in dataloader:
            pixel_values = inputs['pixel_values'].to(device)
            labels = inputs['labels'].to(device)

            input_ids_ctx = inputs['input_ids'].to(device)
            attention_mask_ctx = inputs['attention_mask'].to(device)
            ctx_len = inputs['context_length']
            if isinstance(ctx_len, torch.Tensor):
                ctx_len = ctx_len[0].item()

            input_ids_iso = inputs['input_ids_isolated'].to(device)
            attention_mask_iso = inputs['attention_mask_isolated'].to(device)
            labels_iso = inputs['labels_isolated'].to(device)
            ctx_len_iso = inputs['context_length_isolated']
            if isinstance(ctx_len_iso, torch.Tensor):
                ctx_len_iso = ctx_len_iso[0].item()

            with torch.autocast(
                device_type=device, dtype=torch.float16,
                enabled=use_amp
            ):
                outputs_ctx = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_ctx,
                    attention_mask=attention_mask_ctx,
                    labels=labels,
                    context_length=ctx_len
                )
                outputs_iso = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids_iso,
                    attention_mask=attention_mask_iso,
                    labels=labels_iso,
                    context_length=ctx_len_iso
                )

            loss_net = outputs_ctx.loss + outputs_iso.loss
            val_losses.append(loss_net.item())
            val_ctx_losses.append(outputs_ctx.loss.item())
            val_iso_losses.append(outputs_iso.loss.item())
            val_ctx_accs.append(outputs_ctx.accuracy.item())
            val_iso_accs.append(outputs_iso.accuracy.item())

    avg = lambda lst: sum(lst) / len(lst)
    return {
        'loss': avg(val_losses),
        'ctx_loss': avg(val_ctx_losses),
        'iso_loss': avg(val_iso_losses),
        'ctx_acc': avg(val_ctx_accs),
        'iso_acc': avg(val_iso_accs),
        'avg_acc': (avg(val_ctx_accs) + avg(val_iso_accs)) / 2
    }

print("Evaluation function defined.")

In [ ]:
# ============================================================
# Stage 3: Context-Aware Training Loop (RoPE model)
# ============================================================
# Loss: L = -log(Pc) - log(Pi) + lambda * max(0, loss_ctx - loss_iso)
# ============================================================

s3_train_losses, s3_train_accs = [], []
s3_val_losses, s3_val_accs = [], []
s3_train_ctx_losses, s3_train_iso_losses = [], []
s3_train_ctx_accs, s3_train_iso_accs = [], []
s3_train_margin_terms = []
s3_val_ctx_losses, s3_val_iso_losses = [], []
s3_val_ctx_accs, s3_val_iso_accs = [], []

start_epoch_s3 = 0

print(f"\n{'='*60}")
print(f"  Stage 3: Context-Aware Fine-Tuning (RoPE)")
print(f"  L = -log(Pc) - log(Pi) + {MARGIN_LAMBDA}*max(0, l_ctx - l_iso)")
print(f"{'='*60}\n")

for epoch in range(start_epoch_s3, EPOCHS_S3):
    epoch_losses, epoch_accs = [], []
    epoch_ctx_losses, epoch_iso_losses = [], []
    epoch_ctx_accs, epoch_iso_accs = [], []
    epoch_margin = []

    model_s3.train()
    progress_bar = tqdm.tqdm(
        train_dataloader_ctx,
        total=len(train_dataloader_ctx),
        desc=f'Epoch {epoch + 1}/{EPOCHS_S3}'
    )

    for inputs in progress_bar:
        optimizer_s3.zero_grad()

        pixel_values = inputs['pixel_values'].to(device)
        labels = inputs['labels'].to(device)

        # --- Context path ---
        input_ids_ctx = inputs['input_ids'].to(device)
        attention_mask_ctx = inputs['attention_mask'].to(device)
        ctx_len = inputs['context_length']
        if isinstance(ctx_len, torch.Tensor):
            ctx_len = ctx_len[0].item()

        # --- Isolation path ---
        input_ids_iso = inputs['input_ids_isolated'].to(device)
        attention_mask_iso = inputs['attention_mask_isolated'].to(device)
        labels_iso = inputs['labels_isolated'].to(device)
        ctx_len_iso = inputs['context_length_isolated']
        if isinstance(ctx_len_iso, torch.Tensor):
            ctx_len_iso = ctx_len_iso[0].item()

        with torch.autocast(
            device_type=device, dtype=torch.float16, enabled=use_amp
        ):
            outputs_ctx = model_s3(
                pixel_values=pixel_values,
                input_ids=input_ids_ctx,
                attention_mask=attention_mask_ctx,
                labels=labels,
                context_length=ctx_len,
                return_per_sample_loss=True
            )
            outputs_iso = model_s3(
                pixel_values=pixel_values,
                input_ids=input_ids_iso,
                attention_mask=attention_mask_iso,
                labels=labels_iso,
                context_length=ctx_len_iso,
                return_per_sample_loss=True
            )

        combined_base = outputs_ctx.per_sample_loss + outputs_iso.per_sample_loss
        margin_term = MARGIN_LAMBDA * torch.clamp(
            outputs_ctx.per_sample_loss - outputs_iso.per_sample_loss,
            min=0.0
        )
        weighted_loss = (combined_base + margin_term).mean()

        scaler_s3.scale(weighted_loss).backward()
        scaler_s3.step(optimizer_s3)
        scaler_s3.update()

        epoch_losses.append(weighted_loss.item())
        epoch_ctx_losses.append(outputs_ctx.loss.item())
        epoch_iso_losses.append(outputs_iso.loss.item())
        epoch_ctx_accs.append(outputs_ctx.accuracy.item())
        epoch_iso_accs.append(outputs_iso.accuracy.item())
        epoch_accs.append(
            (outputs_ctx.accuracy.item() + outputs_iso.accuracy.item()) / 2
        )
        epoch_margin.append(margin_term.mean().item())

        progress_bar.set_postfix({
            'loss': f'{weighted_loss.item():.4f}',
            'ctx_acc': f'{outputs_ctx.accuracy.item():.4f}',
            'iso_acc': f'{outputs_iso.accuracy.item():.4f}'
        })

    avg = lambda lst: sum(lst) / len(lst)
    s3_train_losses.append(avg(epoch_losses))
    s3_train_accs.append(avg(epoch_accs))
    s3_train_ctx_losses.append(avg(epoch_ctx_losses))
    s3_train_iso_losses.append(avg(epoch_iso_losses))
    s3_train_ctx_accs.append(avg(epoch_ctx_accs))
    s3_train_iso_accs.append(avg(epoch_iso_accs))
    s3_train_margin_terms.append(avg(epoch_margin))

    val_metrics = evaluate_context_model(
        model_s3, val_dataloader_ctx, device, use_amp
    )
    s3_val_losses.append(val_metrics['loss'])
    s3_val_accs.append(val_metrics['avg_acc'])
    s3_val_ctx_losses.append(val_metrics['ctx_loss'])
    s3_val_iso_losses.append(val_metrics['iso_loss'])
    s3_val_ctx_accs.append(val_metrics['ctx_acc'])
    s3_val_iso_accs.append(val_metrics['iso_acc'])

    print(f"\nEpoch {epoch + 1}/{EPOCHS_S3}:")
    print(f"  Train - Loss: {avg(epoch_losses):.4f} | Avg Acc: {avg(epoch_accs):.4f}")
    print(f"    Ctx Loss: {avg(epoch_ctx_losses):.4f} | Ctx Acc: {avg(epoch_ctx_accs):.4f}")
    print(f"    Iso Loss: {avg(epoch_iso_losses):.4f} | Iso Acc: {avg(epoch_iso_accs):.4f}")
    print(f"    Margin Mean: {avg(epoch_margin):.4f}")
    print(f"  Val   - Loss: {val_metrics['loss']:.4f} | Avg Acc: {val_metrics['avg_acc']:.4f}")
    print(f"    Ctx Loss: {val_metrics['ctx_loss']:.4f} | Ctx Acc: {val_metrics['ctx_acc']:.4f}")
    print(f"    Iso Loss: {val_metrics['iso_loss']:.4f} | Iso Acc: {val_metrics['iso_acc']:.4f}")

    ckpt_name = f'checkpoint_rope_context_epoch_{epoch + 1}.pt'
    save_checkpoint(
        model_s3, optimizer_s3, epoch + 1,
        s3_train_losses[-1], s3_val_losses[-1],
        s3_train_accs[-1], s3_val_accs[-1],
        OUTPUT_DIR_S3, ckpt_name
    )

print("\nStage 3 training complete!")

### 20. Save Stage 3 Final Model

In [ ]:
final_s3_path = os.path.join(OUTPUT_DIR_S3, 'final_rope_context_stage3_model.pth')
save_final_model(model_s3, final_s3_path)
print(f"Stage 3 model saved to: {final_s3_path}")

### 21. Plot Stage 3 Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(s3_train_losses) + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax = axes[0, 0]
ax.plot(epochs_range, s3_train_losses, 'b-o', label='Train Loss (combined)')
ax.plot(epochs_range, s3_val_losses, 'r-o', label='Val Loss (combined)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Combined Dual-Path Loss'); ax.legend(); ax.grid(True)

ax = axes[0, 1]
ax.plot(epochs_range, s3_train_ctx_losses, 'b-o', label='Train Ctx Loss')
ax.plot(epochs_range, s3_train_iso_losses, 'b-s', linestyle='--', label='Train Iso Loss')
ax.plot(epochs_range, s3_val_ctx_losses, 'r-o', label='Val Ctx Loss')
ax.plot(epochs_range, s3_val_iso_losses, 'r-s', linestyle='--', label='Val Iso Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Context vs Isolation Loss'); ax.legend(); ax.grid(True)

ax = axes[1, 0]
ax.plot(epochs_range, s3_train_ctx_accs, 'b-o', label='Train Ctx Acc')
ax.plot(epochs_range, s3_train_iso_accs, 'b-s', linestyle='--', label='Train Iso Acc')
ax.plot(epochs_range, s3_val_ctx_accs, 'r-o', label='Val Ctx Acc')
ax.plot(epochs_range, s3_val_iso_accs, 'r-s', linestyle='--', label='Val Iso Acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Context vs Isolation Accuracy'); ax.legend(); ax.grid(True)

ax = axes[1, 1]
ax.plot(epochs_range, s3_train_margin_terms, 'g-o', label='Margin Term (train)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Mean Margin Penalty')
ax.set_title(f'Margin Penalty (lambda={MARGIN_LAMBDA})'); ax.legend(); ax.grid(True)

plt.suptitle('Stage 3: Context-Aware Fine-Tuning (RoPE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR_S3, 'stage3_rope_training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {plot_path}")

### 22. Stage 2 vs Stage 3 Comparison

In [ ]:
print("=" * 55)
print("  Stage 2 (Abs PE) vs Stage 3 (RoPE + Context)")
print("=" * 55)

print(f"\n{'Metric':<25} {'Stage 2':>10} {'Stage 3':>10}")
print("-" * 50)
print(f"{'Val Loss':<25} {validation_losses[-1]:>10.4f} {s3_val_losses[-1]:>10.4f}")
print(f"{'Val Accuracy':<25} {validation_accuracies[-1]:>10.4f} {s3_val_accs[-1]:>10.4f}")
print(f"{'Val Ctx Accuracy':<25} {'N/A':>10} {s3_val_ctx_accs[-1]:>10.4f}")
print(f"{'Val Iso Accuracy':<25} {'N/A':>10} {s3_val_iso_accs[-1]:>10.4f}")
print(f"{'Train Loss':<25} {train_losses[-1]:>10.4f} {s3_train_losses[-1]:>10.4f}")
print(f"{'Train Accuracy':<25} {train_accuracies[-1]:>10.4f} {s3_train_accs[-1]:>10.4f}")

ctx_benefit = s3_val_ctx_accs[-1] - s3_val_iso_accs[-1]
print(f"\nContext benefit (ctx_acc - iso_acc): {ctx_benefit:+.4f}")
if ctx_benefit > 0:
    print("The model is leveraging context to improve recognition.")
else:
    print("Context is not helping yet — consider more epochs or adjusting lambda.")

### 23. Download Stage 3 Model

In [ ]:
from google.colab import files
files.download(final_s3_path)